# Lab 7 — Go-to-goal: trajectory analysis
Student ID ending: **70** · wheel diameter **5.6 cm** (r = 0.028 m) · base **B = 15.4 cm**

Upload `odometry_data_sim.txt` and `odometry_data_real.txt` (format `theta_l,theta_r,time`, no header) and run all cells.

Import of necessary libraries.

In [ ]:
import numpy as np                                  # library used for working with arrays
import matplotlib.pyplot as plt                     # library for creating visualizations
import pandas as pd                                 # tables
import math as m                                    # mathematical functions
import os

Robot parameters, waypoints of the digits **7** and **0** and the controller settings (the same values as in `go_to_goal_sim.qrs` / `go_to_goal_real.qrs`).

In [ ]:
r = 0.028                                           # wheel radius in meters (diameter 5.6 cm)
B = 0.154                                           # base distance between the wheels in meters

# digit 7: (0,40) -> (25,40) -> (5,0)   gap: (5,0) -> (40,0)   digit 0: (40,0) -> (65,0) -> (65,40) -> (40,40) -> (40,0)   [cm]
X_ref = np.array([0.0, 0.25, 0.05, 0.40, 0.65, 0.65, 0.40, 0.40])   # X coordinates of the waypoints, m
Y_ref = np.array([0.40, 0.40, 0.0, 0.0, 0.0, 0.40, 0.40, 0.0])      # Y coordinates of the waypoints, m
course_0 = 0.0                                      # initial course of the robot (facing +X), rad

K_v, v_0, K_theta, epsilon = 50, 20, 50, 0.03       # gains and acceptance radius used in the robot program

Upload the two log files (in Colab). If they are already in the working folder this cell does nothing.

In [ ]:
need = [f for f in ['odometry_data_sim.txt', 'odometry_data_real.txt'] if not os.path.exists(f)]
if need:
    try:
        from google.colab import files
        print('Please upload:', need)
        files.upload()
    except ImportError:
        print('Put these files next to the notebook:', need)

## Step 1 — Reconstruct the trajectory
`path = (theta_r + theta_l)·r/2`, `course = course_0 + (theta_r − theta_l)·r/B`, `x[k] = x[k−1] + Δpath·cos(course)`, `y[k] = y[k−1] + Δpath·sin(course)`

In [ ]:
def load_log(name):
    data = np.loadtxt(name, delimiter=',')          # theta_l, theta_r (rad), time (ms)
    restart = np.where(np.diff(data[:, 2]) < 0)[0]  # the program appends: keep only the last run
    if len(restart):
        data = data[restart[-1] + 1:]
    theta_l, theta_r, time_ms = data[:, 0], data[:, 1], data[:, 2]
    t = (time_ms - time_ms[0]) / 1000               # time in seconds from the first sample
    return theta_l, theta_r, t

def reconstruct(theta_l, theta_r):
    n = len(theta_l)
    path = (theta_r + theta_l) * r / 2              # distance traveled
    course = course_0 + (theta_r - theta_l) * r / B # robot course angle
    x = np.zeros(n); y = np.zeros(n)
    x[0], y[0] = X_ref[0], Y_ref[0]                 # start from the first waypoint
    for k in range(1, n):
        x[k] = x[k-1] + (path[k] - path[k-1]) * m.cos(course[k])
        y[k] = y[k-1] + (path[k] - path[k-1]) * m.sin(course[k])
    return x, y, course

runs = {}
for name, file in [('sim', 'odometry_data_sim.txt'), ('real', 'odometry_data_real.txt')]:
    theta_l, theta_r, t = load_log(file)
    x, y, course = reconstruct(theta_l, theta_r)
    runs[name] = dict(t=t, x=x, y=y, course=course)
    print(f'{name}: {len(t)} samples, {t[-1]:.1f} s')

## Step 2 — Ideal trajectory
Segment `i` goes from waypoint `i−1` to waypoint `i`: `course_ref[i] = atan2(Y_ref[i] − Y_ref[i−1], X_ref[i] − X_ref[i−1])`

In [ ]:
N = len(X_ref)
course_ref = np.zeros(N)
for i in range(1, N):
    course_ref[i] = m.atan2(Y_ref[i] - Y_ref[i-1], X_ref[i] - X_ref[i-1])
print('course_ref, deg:', np.round(np.degrees(course_ref[1:]), 1))

## Step 3 and Step 4 — Assign samples to segments, `distance` and `courseAngle`
Each sample may stay on the current segment or move to the next one (never back).

`distance = −(x − X_ref[i−1])·sin(course_ref[i]) + (y − Y_ref[i−1])·cos(course_ref[i])` (positive = left of the segment)
`courseAngle = course_ref[i] − course`, wrapped to [−180°, 180°]

In [ ]:
def dist_to_segment(px, py, i):                     # distance from a point to segment i (waypoint i-1 -> i)
    ax, ay, bx, by = X_ref[i-1], Y_ref[i-1], X_ref[i], Y_ref[i]
    vx, vy = bx - ax, by - ay
    s = ((px - ax) * vx + (py - ay) * vy) / (vx * vx + vy * vy)
    s = min(max(s, 0), 1)
    return m.hypot(px - (ax + s * vx), py - (ay + s * vy))

def wrap_deg(a):
    return (a + 180) % 360 - 180

def errors(run):
    x, y, course = run['x'], run['y'], run['course']
    n = len(x)
    seg = np.zeros(n, dtype=int); distance = np.zeros(n); courseAngle = np.zeros(n)
    i = 1
    for k in range(n):
        if i < N - 1 and dist_to_segment(x[k], y[k], i + 1) < dist_to_segment(x[k], y[k], i):
            i = i + 1                               # move to the next segment
        seg[k] = i
        distance[k] = -(x[k] - X_ref[i-1]) * m.sin(course_ref[i]) + (y[k] - Y_ref[i-1]) * m.cos(course_ref[i])
        courseAngle[k] = wrap_deg(m.degrees(course_ref[i] - course[k]))
    run['seg'], run['distance'], run['courseAngle'] = seg, distance, courseAngle

for name in runs:
    errors(runs[name])

## Step 5 — Plots and summary
### 1. Ideal, simulator and real trajectory

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(X_ref, Y_ref, 'k--', linewidth=2, label='Ideal')
plt.plot(X_ref, Y_ref, 'go', markersize=7, label='Waypoints')
for k in range(N):
    plt.text(X_ref[k] + 0.01, Y_ref[k] + 0.01, str(k))
plt.plot(runs['sim']['x'], runs['sim']['y'], 'b-', label='Simulator')
plt.plot(runs['real']['x'], runs['real']['y'], 'r-', label='Real robot (odometry)')
plt.xlabel('X (m)'); plt.ylabel('Y (m)'); plt.title('ID 70: go-to-goal trajectories')
plt.axis('equal'); plt.grid(True); plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

### 2. `distance(t)` and `courseAngle(t)`

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name, c in [('sim', 'b'), ('real', 'r')]:
    ax[0].plot(runs[name]['t'], runs[name]['distance'] * 100, c, label=name)
    ax[1].plot(runs[name]['t'], runs[name]['courseAngle'], c, label=name)
ax[0].set_ylabel('distance (cm)'); ax[0].grid(True); ax[0].legend()
ax[1].set_ylabel('courseAngle (deg)'); ax[1].set_xlabel('t (s)'); ax[1].grid(True); ax[1].legend()
plt.show()

### 3. Waypoint miss

In [ ]:
miss = pd.DataFrame({'X_ref (cm)': X_ref * 100, 'Y_ref (cm)': Y_ref * 100})
for name in runs:
    x, y = runs[name]['x'], runs[name]['y']
    miss[f'miss {name} (cm)'] = [np.min(np.hypot(x - X_ref[k], y - Y_ref[k])) * 100 for k in range(N)]
miss.round(2)

### 4. Summary

In [ ]:
summary = {}
for name in runs:
    R = runs[name]
    summary[name] = {
        'max |distance| (cm)': np.max(np.abs(R['distance'])) * 100,
        'RMS distance (cm)': np.sqrt(np.mean(R['distance'] ** 2)) * 100,
        'mean |courseAngle| (deg)': np.mean(np.abs(R['courseAngle'])),
        'final error to last waypoint (cm)': m.hypot(R['x'][-1] - X_ref[-1], R['y'][-1] - Y_ref[-1]) * 100,
        'run time (s)': R['t'][-1],
    }
pd.DataFrame(summary).round(2)

Per-segment errors (helps to answer which segments / corners are bad).

In [ ]:
rows = []
for name in runs:
    R = runs[name]
    for i in range(1, N):
        s = R['seg'] == i
        if s.any():
            rows.append({'run': name, 'segment': f'{i-1}->{i}',
                         'max |distance| (cm)': np.max(np.abs(R['distance'][s])) * 100,
                         'mean |courseAngle| (deg)': np.mean(np.abs(R['courseAngle'][s]))})
pd.DataFrame(rows).round(2)

## Measured end point
Measure the final position of the real robot with a tape measure (in cm, in the same frame: start point = (0, 40), X along the first segment of the 7) and write it below.

In [ ]:
x_meas, y_meas = 40.0, 0.0                          # <-- PUT YOUR TAPE-MEASURE VALUES HERE (cm)

x_odo, y_odo = runs['real']['x'][-1] * 100, runs['real']['y'][-1] * 100
print(f'Last waypoint:           ({X_ref[-1]*100:.1f}, {Y_ref[-1]*100:.1f}) cm')
print(f'Odometry end point:      ({x_odo:.1f}, {y_odo:.1f}) cm')
print(f'Measured end point:      ({x_meas:.1f}, {y_meas:.1f}) cm')
print(f'Measured vs last waypoint: {m.hypot(x_meas - X_ref[-1]*100, y_meas - Y_ref[-1]*100):.1f} cm')
print(f'Measured vs odometry:      {m.hypot(x_meas - x_odo, y_meas - y_odo):.1f} cm')

## Conclusions (3–6 sentences)
*Fill in after looking at your plots.*

1. **Where do `distance` and `courseAngle` grow?** … (e.g. the diagonal of the 7 (segment 1→2) and the sharp corner at waypoint 2, where the robot has to turn ≈117°; with `K_θ = 50` and `ε = 3 cm` the corner is …)
2. **Control or localization?** … (if the odometry trajectory of the real robot is close to the ideal one but the tape-measured end point is far from it, the gap is a **localization** problem (wheel slip, wrong r/B); if the odometry trajectory itself deviates, it is a **control** problem.)